# Python Math

> 📘 **Python Mastery** · Module 05 — Intermediate Python · Lesson 2/7

Beyond `+` and `*`, Python hides a full maths toolkit: built-ins for everyday sums, a `math` module for serious numbers, exact arithmetic for money, and controlled randomness.

## 🎯 Learning Objectives

- Use the built-ins `abs`, `round`, `pow`, `divmod`, `min`, `max` and `sum`
- Explain why `round(2.5)` returns **2** (banker's rounding)
- Choose correctly between `ceil`, `floor`, `trunc` and `round`
- Compare floats safely with `math.isclose` instead of `==`
- Do money math with `decimal.Decimal`
- Generate reproducible randomness with `random.seed`

## 1. Built-in Helpers: `abs`, `pow`, `divmod`

Three workhorses you'll reach for weekly:

- `abs(x)` — distance from zero; turns `-7` into `7`.
- `pow(b, e)` — same as `b ** e`; it even accepts a third argument for modular exponentiation, beloved by cryptography: `pow(b, e, m)`.
- `divmod(a, b)` — division with remainder answered in one call: returns the tuple `(a // b, a % b)`.

**Syntax:**
```python
abs(-7)          # -> 7
pow(2, 10)       # -> 1024
divmod(47, 6)    # -> (7, 5)
```

**Example:** splitting students into groups.

In [ ]:
print(abs(-27))              # temperature drop of -27 degrees
print(pow(2, 16))            # 2^16
print(pow(7, 3, 100))        # (7^3) mod 100 -- one step, no huge intermediate

students, groups = 47, 6     # divmod shines for splitting
per_group, leftover = divmod(students, groups)
print(f"{per_group} per group, {leftover} left over")

## 2. Aggregates: `min`, `max`, `sum`

The three great reducers. They accept any iterable, and `min`/`max` take an optional `key=` function so you can compare by *some property* instead of raw value.

```python
min(iterable), max(iterable), sum(iterable)
min(data, key=func)      # compare items using func(item)
sum(items, start)        # add `start` to the total first
```

**Example:** shopping cart analytics.

In [ ]:
cart = [
    ("rice 5kg",   420),
    ("eggs dozen", 155),
    ("milk 1L",     95),
]

prices = [p for _, p in cart]
print("cheapest:", min(prices))
print("dearest :", max(prices))
print("total   :", sum(prices))

print(max(cart, key=lambda item: item[1]))   # whole tuple, judged by price
print(sum(prices, 100))                      # start from 100 (e.g. a delivery fee)

## 3. `round()` and the Banker's-Rounding Gotcha

`round(x)` rounds to the nearest integer, `round(x, n)` to `n` decimal places. But here is the surprise every Python learner hits:

```python
round(2.5)   # 2  ... not 3!
round(0.5)   # 0
round(1.5)   # 2
round(3.5)   # 4
```

Python uses **banker's rounding**: exact halfway cases go to the nearest **even** number. The idea is that always-rounding-up (the school rule) slowly inflates totals across millions of transactions; rounding half-to-even is unbiased on average.

> 🔍 **Under the Hood:** floats are IEEE 754 binary fractions, so most decimals can't be stored exactly. `2.675` is actually stored as `2.67499999999999982...`, which is why `round(2.675, 2)` gives `2.67`, not `2.68`. The tie isn't really a tie — the stored value already leans down.

**Syntax:**
```python
round(x)         # nearest integer; ties go to EVEN
round(x, n)      # n decimal places
round(x, -k)     # negative n rounds to tens (-1), hundreds (-2), thousands (-3)
```

**Example:**

In [ ]:
print(round(2.5), round(0.5), round(1.5), round(3.5))   # ties go to EVEN
print(round(-0.5))                                       # even neighbour again -> 0

print(round(2.675, 2))       # 2.67 -- the stored value leans low
print(round(15842, -3))      # nearest thousand -> 16000
print(round(92.4567, 2))     # ordinary rounding works fine

## 4. The `math` Module: Constants

`math` ships physics-grade constants plus dozens of functions. Four worth memorising:

| Constant | Meaning |
|---|---|
| `math.pi` | pi ~ 3.14159 |
| `math.e` | Euler's number ~ 2.71828 |
| `math.tau` | tau = 2*pi ~ 6.28318 - the 'full turn' constant |
| `math.inf` / `math.nan` | infinity / not-a-number |

**Syntax:**
```python
import math
math.pi          # 3.141592653589793
math.tau / 2 == math.pi
```

**Example:** circle geometry -- notice how `tau` makes full-turn formulas cleaner.

In [ ]:
import math

r = 7
print(f"pi  = {math.pi:.5f}")
print(f"tau = {math.tau:.5f}  (= 2*pi? {math.tau == 2 * math.pi})")

circumference = math.tau * r           # one full turn around radius r
print(f"circumference = {circumference:.2f}")

print(math.inf > 10**300)              # bigger than any finite float

## 5. Powers and Roots: `sqrt`, `isqrt`, `pow`

- `math.sqrt(x)` -- square root, always returns a **float** (`ValueError` for negatives).
- `math.isqrt(n)` -- **integer** square root: the floor of the true root, computed exactly with integers, so no float fuzz even on huge numbers.
- Built-in `pow(b, e)` keeps integer types for integer inputs; `math.pow(b, e)` always returns float.

**Syntax:**
```python
math.sqrt(16)     # 4.0
math.isqrt(17)    # 4   (largest int whose square <= 17)
pow(2, 10)        # 1024      int
math.pow(2, 10)   # 1024.0    float
```

**Example:** testing perfect squares without float error.

In [ ]:
import math

n = 10_000_049
root_int = math.isqrt(n)
print(root_int, "-> perfect square?", root_int * root_int == n)

print(math.sqrt(17))                 # 4.123105625617661 (approximate!)
print(math.isqrt(17))                # 4 (exact floor)

big = 10 ** 40 + 3
print(math.isqrt(big))               # exact integer floor even at 40 digits
print(math.sqrt(big))                # float can only approximate: 1e+20

## 6. `ceil` vs `floor` vs `trunc` vs `round`

Four ways to squash a float onto the integer line -- they disagree especially on negatives:

| Function | Direction | `-2.7` | `2.7` | Typical use |
|---|---|---|---|---|
| `math.ceil` | up (towards +inf) | -2 | 3 | "how many buses do we need?" |
| `math.floor` | down (towards -inf) | -3 | 2 | complete pairs, page counts |
| `math.trunc` | chop towards zero | -2 | 2 | what `int()` does |
| `round` | nearest (ties -> even) | -3 | 3 | general display |

**Syntax:**
```python
math.ceil(2.1)    # 3
math.floor(2.9)   # 2
math.trunc(-2.9)  # -2
```

**Example:** the classic bus problem.

In [ ]:
import math

people, seats = 53, 20
buses = math.ceil(people / seats)
print(f"{people} people need {buses} buses")     # 2.65 -> must round UP

for x in (-2.7, 2.7):
    print(f"x={x:>4}: ceil={math.ceil(x)}  floor={math.floor(x)}  "
          f"trunc={math.trunc(x)}  round={round(x)}")

## 7. Combinatorics: `factorial`, `gcd`, `lcm`, `comb`

A pocket toolbox for counting problems:

| Function | Answers the question | Example |
|---|---|---|
| `math.factorial(n)` | ways to arrange n items | `factorial(6) = 720` |
| `math.gcd(a, b)` | largest shared divisor | simplify fractions |
| `math.lcm(a, b)` | smallest shared multiple | "when do both alarms ring?" |
| `math.comb(n, k)` | ways to choose k from n (order ignored) | lottery hands |
| `math.perm(n, k)` | ordered selections | podium finishes |

**Syntax:**
```python
math.factorial(6)   # 720
math.gcd(48, 180)   # 12
math.comb(49, 6)    # 13983816
```

**Example:**

In [ ]:
import math

print(math.factorial(6), "-> line-ups of 6 friends")
g = math.gcd(48, 180)
print(g, "= gcd; 48/180 simplifies to", 48 // g, "/", 180 // g)
print(math.lcm(4, 6), "- alarms ring together every", math.lcm(4, 6), "minutes")
print(math.comb(49, 6), "possible lottery tickets")     # order doesn't matter
print(math.perm(10, 3), "gold-silver-bronze outcomes")  # order matters

## 8. Why `0.1 + 0.2 != 0.3` -- and the Fix

Run this and watch Python betray your schoolteacher:

```python
0.1 + 0.2 == 0.3        # False!
0.1 + 0.2               # 0.30000000000000004
```

> 🔍 **Under the Hood:** floats are stored as binary fractions (IEEE 754 doubles). `0.1` has no finite binary form -- like 1/3 in decimal -- so Python stores the *nearest* representable value. Two near-misses added together stay a near-miss: `0.30000000000000004`. Every language does this; Python just refuses to hide it.

Never compare floats with `==`. Use `math.isclose(a, b)`:

```python
math.isclose(a, b)                # relative tolerance (default rel_tol=1e-09)
math.isclose(a, b, abs_tol=1e-9)  # absolute tolerance -- needed near ZERO
```

Related rescue: `math.fsum(values)` adds floats while tracking precision, so long sums don't drift.

**Example:**

In [ ]:
import math

print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)                              # False -- shock!
print(math.isclose(0.1 + 0.2, 0.3))                  # True  -- the fix

print(sum([0.1] * 10) == 1.0)                        # accumulation error
print(math.fsum([0.1] * 10) == 1.0)                  # fsum tracks precision
print(math.isclose(1e-12, 0.0, abs_tol=1e-9))        # tiny values need abs_tol

## 9. Money Math with `decimal.Decimal`

Floats are perfect for physics, terrible for invoices. The `decimal` module stores base-10 numbers exactly and lets you control rounding -- which is what accounting standards demand.

Two rules:
1. Build `Decimal` from a **string**, never from a float (`Decimal("19.99")`, not `Decimal(19.99)`).
2. Round explicitly with `.quantize()`.

**Syntax:**
```python
from decimal import Decimal, ROUND_HALF_UP
price = Decimal("19.99")
total = price * 3
total.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
```

**Example:** the invoice test that floats fail.

In [ ]:
from decimal import Decimal, ROUND_HALF_UP

print(19.99 * 3)                                     # 59.97000000000001 ?!
print(Decimal("19.99") * 3)                          # exactly 59.97

price = Decimal("4500.00")
vat = price * Decimal("0.15")
print("VAT:", vat.quantize(Decimal("0.01")))         # 675.00, clean

print(round(Decimal("2.5")))                         # banker's again -> 2
print(Decimal("2.5").quantize(Decimal("1"), rounding=ROUND_HALF_UP))  # money rule -> 3

print(Decimal(0.1))                                  # the trap: built FROM a float

## 10. Randomness with `random`

The `random` module produces pseudo-random numbers. Key tools:

| Call | Gives |
|---|---|
| `random.seed(42)` | fixes the sequence -- same numbers on every run |
| `random.randint(1, 6)` | inclusive dice roll |
| `random.choice(seq)` | one item |
| `random.shuffle(lst)` | shuffles **in place** (returns None!) |
| `random.sample(seq, k)` | k unique items, original untouched |
| `random.uniform(0, 1)` | continuous value between 0 and 1 |

> 🔍 **Under the Hood:** `random` uses the Mersenne Twister generator. Given the same seed it replays the identical sequence forever -- that's why seeding makes experiments reproducible, and why `random` must never guard passwords (use the `secrets` module for security).

**Syntax:**
```python
import random
random.seed(42)
random.randint(1, 6)
```

**Example:** game night, deterministically.

In [ ]:
import random

random.seed(42)                       # reproducible from here on

print([random.randint(1, 6) for _ in range(5)])      # five dice rolls
print(random.choice(["rock", "paper", "scissors"]))

deck = list(range(1, 11))
random.shuffle(deck)                  # mutates deck itself, returns None
print(deck[:5], "... shuffled in place")

winners = random.sample(["Sara", "Arif", "Nadia", "Rafi", "Mina"], 2)
print("raffle winners:", winners)

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| `0.1 + 0.2 == 0.3` | `False` -- binary storage noise | `math.isclose(0.1 + 0.2, 0.3)` |
| Using `round()` for invoices | banker's rounding + float noise break accounting | `Decimal(...).quantize(..., ROUND_HALF_UP)` |
| `Decimal(0.1)` | builds Decimal *from the imprecise float* | `Decimal("0.1")` from a string |
| `math.sqrt(-1)` | `ValueError` -- real numbers only | `cmath.sqrt(-1)` for complex results |
| Expecting `shuffle()` to return the list | returns `None`; it mutates in place | shuffle, then use the same list |
| Summing many floats with `sum` | error accumulates step by step | `math.fsum(values)` |

## 💡 Best Practices & Pro Tips

- Floats for science, `Decimal` for money -- decide per domain, never mix them in one column.
- Any numeric unit test should assert closeness, not equality (`math.isclose`, or `pytest.approx`).
- Prefer integers end-to-end when possible (`isqrt`, `//`, counts) -- they are immune to all of this.
- Seed once at the top of an experiment script; reproducibility is a feature, not a chore.
- **AI-engineering relevance:** NumPy vectorises this entire lesson over millions of values at C speed. Loss curves are floats -- compare them with tolerances. And `random.seed` together with `np.random.seed` / `torch.manual_seed` is what makes a training run replayable.

## 📌 Summary

| Tool | What it does | Example |
|---|---|---|
| `abs / pow / divmod` | magnitude, power, quotient+remainder | `divmod(47, 6) -> (7, 5)` |
| `min / max / sum` | aggregates; `key=` for custom ranking | `max(cart, key=lambda x: x[1])` |
| `round(x, n)` | rounds; ties go to the EVEN neighbour | `round(2.5) -> 2` |
| `math.ceil / floor / trunc` | up / down / towards zero | `math.ceil(53/20) -> 3` |
| `math.factorial / gcd / lcm / comb` | counting helpers | `math.gcd(48, 180) -> 12` |
| `math.isclose(a, b)` | safe float comparison | `isclose(0.1+0.2, 0.3)` |
| `Decimal("19.99")` | exact base-10 arithmetic | `.quantize(Decimal("0.01"))` |
| `random.seed / randint / choice / shuffle / sample` | reproducible randomness | `seed(42); randint(1, 6)` |

Key takeaways:

- `round(2.5) == 2` is not a bug -- it's ties-to-even on imperfectly stored binary fractions.
- Never test floats with `==`; use `math.isclose` (with `abs_tol` near zero).
- Money means `Decimal`, built from strings, rounded with explicit rules.
- Seed your randomness or your results can never be reproduced.

## 🔗 Next Lesson

Up next: **[03_Datetime](../03_Datetime/notes.ipynb)** -- dates, times, deadlines and the arithmetic between them.